# Silver Layer — Limpeza, Tipagem e Integração

**Tech Challenge – Fase 2 | PosTech FIAP**

A Bronze armazena tudo como `str` sem transformações. Aqui aplicamos:

| Transformação | O que faz |
|---|---|
| Cast de tipos | `str` → `Int64` (ano, serie, rede) e `float64` (métricas) |
| Drop de metadados | Remove `_ingestion_timestamp`, `_source_file`, `_pipeline_version` |
| Nulos | Mantidos como `NaN` — ausência de dado é informação válida |
| Join | `indicador_*` + `meta_*` formam tabelas consolidadas por município e UF |

In [1]:
import pandas as pd
from pathlib import Path

BRONZE_DIR = Path("..") / "layers" / "bronze"
SILVER_DIR = Path("..") / "layers" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

META_COLS = ["_ingestion_timestamp", "_source_file", "_pipeline_version"]

## 1. Leitura da Bronze

5 arquivos Parquet. Todos com colunas `str` e 3 colunas de metadados de ingestão que serão removidas.

In [2]:
ind_mun  = pd.read_parquet(BRONZE_DIR / "indicador_municipio.parquet")
ind_uf   = pd.read_parquet(BRONZE_DIR / "indicador_uf.parquet")
meta_br  = pd.read_parquet(BRONZE_DIR / "meta_brasil.parquet")
meta_mun = pd.read_parquet(BRONZE_DIR / "meta_municipio.parquet")
meta_uf  = pd.read_parquet(BRONZE_DIR / "meta_uf.parquet")

for name, df in [("ind_mun", ind_mun), ("ind_uf", ind_uf),
                 ("meta_br", meta_br), ("meta_mun", meta_mun), ("meta_uf", meta_uf)]:
    print(f"  {name:<12} {df.shape}  colunas: {df.shape[1]}")

  ind_mun      (23995, 18)  colunas: 18
  ind_uf       (145, 18)  colunas: 18
  meta_br      (3, 14)  colunas: 14
  meta_mun     (10704, 16)  colunas: 16
  meta_uf      (54, 15)  colunas: 15


## 2. Funções de limpeza

- `drop_meta` — remove as 3 colunas de rastreabilidade Bronze
- `cast_int` — converte para `Int64` (nullable integer — aceita `NaN` sem virar float)
- `cast_numeric` — converte para `float64`; valores não-numéricos viram `NaN`

In [3]:
def drop_meta(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=[c for c in META_COLS if c in df.columns])


def cast_numeric(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def cast_int(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return df

## 3. Silver — indicador_municipio

**Entrada:** 23.995 registros, 18 colunas `str` (Bronze)  
**Saída esperada:** 23.995 registros, 15 colunas tipadas — 3 metadados removidos

As 9 colunas `proporcao_aluno_nivel_*` têm ~48% de nulos. Isso é esperado: municípios muito pequenos não têm alunos suficientes para calcular a distribuição por nível de proficiência.

In [4]:
nivel_cols = [f"proporcao_aluno_nivel_{i}" for i in range(9)]
float_cols = ["taxa_alfabetizacao", "media_portugues"] + nivel_cols

s_ind_mun = (
    ind_mun
    .pipe(drop_meta)
    .pipe(cast_int, ["ano", "serie", "rede"])
    .pipe(cast_numeric, float_cols)
)

print(f"Shape: {s_ind_mun.shape}")
print(f"Nulos taxa_alfabetizacao: {s_ind_mun['taxa_alfabetizacao'].isnull().sum()}")
print(f"Nulos proporcao_nivel_*:  {s_ind_mun['proporcao_aluno_nivel_0'].isnull().sum()} "
      f"({s_ind_mun['proporcao_aluno_nivel_0'].isnull().mean():.0%})")
s_ind_mun.dtypes

Shape: (23995, 15)
Nulos taxa_alfabetizacao: 0
Nulos proporcao_nivel_*:  11547 (48%)


ano                          Int64
id_municipio                   str
serie                        Int64
rede                         Int64
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64
proporcao_aluno_nivel_1    float64
proporcao_aluno_nivel_2    float64
proporcao_aluno_nivel_3    float64
proporcao_aluno_nivel_4    float64
proporcao_aluno_nivel_5    float64
proporcao_aluno_nivel_6    float64
proporcao_aluno_nivel_7    float64
proporcao_aluno_nivel_8    float64
dtype: object

## 4. Silver — indicador_uf

**Entrada:** 145 registros, 18 colunas `str` (Bronze)  
**Saída esperada:** 145 registros, 15 colunas — mesma estrutura que `indicador_municipio`, mas com `sigla_uf` no lugar de `id_municipio`

In [5]:
s_ind_uf = (
    ind_uf
    .pipe(drop_meta)
    .pipe(cast_int, ["ano", "serie", "rede"])
    .pipe(cast_numeric, float_cols)
)

print(f"Shape: {s_ind_uf.shape}")
s_ind_uf.head(3)

Shape: (145, 15)


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Silver — meta_brasil

**Entrada:** 3 registros — uma linha por ano de referência (2023, 2024, 2025)  
**Saída esperada:** mesma estrutura com colunas numéricas tipadas

Cada linha contém a taxa observada no `ano` e as metas projetadas de 2024 a 2030 para atingir 100% de alfabetização nacional.

In [6]:
meta_float_cols = [
    "taxa_alfabetizacao", "percentual_participacao",
    "meta_alfabetizacao_2024", "meta_alfabetizacao_2025", "meta_alfabetizacao_2026",
    "meta_alfabetizacao_2027", "meta_alfabetizacao_2028", "meta_alfabetizacao_2029",
    "meta_alfabetizacao_2030",
]

s_meta_br = (
    meta_br
    .pipe(drop_meta)
    .pipe(cast_int, ["ano", "rede"])
    .pipe(cast_numeric, meta_float_cols)
)

print(f"Shape: {s_meta_br.shape}")
s_meta_br

Shape: (3, 11)


,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2025,<NA>,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80,88.00
1,2024,<NA>,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80,87.37
2,2023,<NA>,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80,86.00


## 6. Silver — meta_municipio

**Entrada:** 10.704 registros — metas por município, ano e rede de ensino  
**Saída esperada:** mesma contagem, colunas numéricas tipadas

`nivel_alfabetizacao` é uma classificação ordinal (0–5) do município em relação ao indicador — mantida como `str` pois é uma categoria, não um valor contínuo.  
Os nulos nas colunas `taxa_*` e `meta_2024` indicam municípios que ainda não tinham resultado disponível no ano de referência.

In [7]:
s_meta_mun = (
    meta_mun
    .pipe(drop_meta)
    .pipe(cast_int, ["ano", "rede"])
    .pipe(cast_numeric, meta_float_cols)
)

print(f"Shape: {s_meta_mun.shape}")
print(f"Valores únicos nivel_alfabetizacao: {sorted(s_meta_mun['nivel_alfabetizacao'].dropna().unique().tolist())}")
print(f"Nulos taxa_alfabetizacao: {s_meta_mun['taxa_alfabetizacao'].isnull().sum()}")
s_meta_mun.head(3)

Shape: (10704, 13)
Valores únicos nivel_alfabetizacao: ['0', '1', '2', '3', '4', '5']
Nulos taxa_alfabetizacao: 120


,ano,id_municipio,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao
0,2023,4301750,<NA>,NaN,NaN,14.05,23.65,37.0,52.68,67.85,80,NaN,NaN
1,2024,4301750,<NA>,4.40,NaN,14.05,23.65,37.0,52.68,67.85,80,0,92.59
2,2024,2406908,<NA>,42.86,7.94,14.05,23.65,37.0,52.68,67.85,80,1,84.00


## 7. Silver — meta_uf

**Entrada:** 54 registros — metas por UF e ano  
**Saída esperada:** mesma contagem com tipos corretos

Os nulos nas colunas de meta indicam UFs para as quais a trajetória ainda não foi definida no período de referência.

In [8]:
s_meta_uf = (
    meta_uf
    .pipe(drop_meta)
    .pipe(cast_int, ["ano", "rede"])
    .pipe(cast_numeric, meta_float_cols)
)

print(f"Shape: {s_meta_uf.shape}")
s_meta_uf.head(3)

Shape: (54, 12)


,ano,sigla_uf,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2024,RR,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,RR,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,AC,<NA>,NaN,NaN,56.9,62.2,67.3,72.0,76.2,80.0,NaN


## 8. Join — municipio_consolidado

Enriquece `indicador_municipio` com as metas projetadas de `meta_municipio`.

**Chave:** `[id_municipio, ano]`  
> A coluna `rede` não é usada como chave porque as duas tabelas usam codificações diferentes — `indicador` usa código numérico (3 = Municipal) e `meta` usa descrição textual ('Municipal'). O join por `id_municipio + ano` já identifica unicamente cada registro.

**Resultado esperado:** 23.995 linhas, 25 colunas — cada indicador municipal enriquecido com metas até 2030 onde disponível (~96% de match).

In [9]:
meta_mun_renamed = s_meta_mun.rename(columns={
    "taxa_alfabetizacao": "taxa_alfabetizacao_meta_base",
    "percentual_participacao": "percentual_participacao_meta",
})

municipio_consolidado = s_ind_mun.merge(
    meta_mun_renamed,
    on=["id_municipio", "ano"],
    how="left",
)

match_rate = municipio_consolidado["meta_alfabetizacao_2030"].notna().mean()
print(f"ind_mun:               {s_ind_mun.shape}")
print(f"municipio_consolidado: {municipio_consolidado.shape}")
print(f"Match rate (meta 2030 preenchida): {match_rate:.1%}")
municipio_consolidado[["id_municipio", "ano", "taxa_alfabetizacao",
                        "meta_alfabetizacao_2030", "nivel_alfabetizacao"]].head(5)

ind_mun:               (23995, 15)
municipio_consolidado: (23995, 26)
Match rate (meta 2030 preenchida): 96.5%


,id_municipio,ano,taxa_alfabetizacao,meta_alfabetizacao_2030,nivel_alfabetizacao
0,1100031,2023,69.10,80.0,3
1,1100072,2023,58.20,80.0,2
2,1100189,2023,69.73,80.0,3
3,1101609,2023,50.70,80.0,2
4,1101807,2023,55.69,80.0,2


## 9. Join — uf_consolidado

Mesmo processo para o nível de UF.

**Chave:** `[sigla_uf, ano]`  
**Resultado esperado:** 145 linhas, 24 colunas — match de 100% pois todas as UFs têm entrada em `meta_uf`.

In [10]:
meta_uf_renamed = s_meta_uf.rename(columns={
    "taxa_alfabetizacao": "taxa_alfabetizacao_meta_base",
    "percentual_participacao": "percentual_participacao_meta",
})

uf_consolidado = s_ind_uf.merge(
    meta_uf_renamed,
    on=["sigla_uf", "ano"],
    how="left",
)

match_rate = uf_consolidado["meta_alfabetizacao_2030"].notna().mean()
print(f"ind_uf:        {s_ind_uf.shape}")
print(f"uf_consolidado: {uf_consolidado.shape}")
print(f"Match rate (meta 2030 preenchida): {match_rate:.1%}")
uf_consolidado[["sigla_uf", "ano", "taxa_alfabetizacao",
                "meta_alfabetizacao_2030"]].head(5)

ind_uf:        (145, 15)
uf_consolidado: (145, 25)
Match rate (meta 2030 preenchida): 100.0%


,sigla_uf,ano,taxa_alfabetizacao,meta_alfabetizacao_2030
0,AM,2023,49.20,80.0
1,PB,2023,55.23,80.0
2,PR,2023,73.12,80.0
3,AP,2023,41.87,80.0
4,PE,2023,58.95,80.0


## 10. Gravação em Parquet

7 arquivos Silver — 5 tabelas limpas individualmente + 2 tabelas consolidadas (join).  
O tamanho final é ligeiramente menor que o Bronze porque tipos nativos (`float64`, `Int64`) comprimem melhor que strings.

In [11]:
outputs = {
    "indicador_municipio":   s_ind_mun,
    "indicador_uf":          s_ind_uf,
    "meta_brasil":           s_meta_br,
    "meta_municipio":        s_meta_mun,
    "meta_uf":               s_meta_uf,
    "municipio_consolidado": municipio_consolidado,
    "uf_consolidado":        uf_consolidado,
}

print(f"{'Arquivo':<35} {'Shape':>15} {'Tamanho':>10}")
print("-" * 63)

for name, df in outputs.items():
    path = SILVER_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False, engine="pyarrow")
    size_kb = path.stat().st_size / 1024
    print(f"{name + '.parquet':<35} {str(df.shape):>15} {size_kb:>8.1f} KB")

Arquivo                                       Shape    Tamanho
---------------------------------------------------------------
indicador_municipio.parquet             (23995, 15)    484.6 KB
indicador_uf.parquet                      (145, 15)     16.2 KB
meta_brasil.parquet                         (3, 11)      7.6 KB
meta_municipio.parquet                  (10704, 13)    215.9 KB
meta_uf.parquet                            (54, 12)      9.8 KB
municipio_consolidado.parquet           (23995, 26)    849.9 KB
uf_consolidado.parquet                    (145, 25)     24.9 KB


## 11. Validação

Verifica duplicatas e percentual médio de nulos por tabela.  
O null médio elevado em `municipio_consolidado` (~33%) vem principalmente das colunas `proporcao_aluno_nivel_*` — esperado conforme identificado na seção 3.

In [12]:
print(f"{'Tabela':<35} {'Duplicatas':>12} {'Null médio':>12} {'Status':>8}")
print("-" * 72)

for name, df in outputs.items():
    dup = df.duplicated().sum()
    null_pct = df.isnull().mean().mean() * 100
    status = "OK" if dup == 0 else f"WARN ({dup} dup)"
    print(f"{name:<35} {dup:>12} {null_pct:>11.1f}% {status:>8}")

Tabela                                Duplicatas   Null médio   Status
------------------------------------------------------------------------


indicador_municipio                            0        28.9%       OK
indicador_uf                                   0        29.0%       OK
meta_brasil                                    0         9.1%       OK


meta_municipio                                 0         8.1%       OK
meta_uf                                        0        12.3%       OK


municipio_consolidado                          0        22.0%       OK
uf_consolidado                                 0        21.5%       OK
